# Notebook Quick Start

See the code and its output together. This notebook demonstrates two display
settings: a compact summary after each cell and one full report at the end.
Both use the default Trace backend with driver memory collection.

Open the downloaded notebook in an environment with `linescope[notebook]`
installed, then run the cells from top to bottom. From a repository checkout:

```console
cd examples/notebooks
uv run --group demo --extra notebook jupyter lab notebook_example.ipynb
```

## Cell summaries

Start with `display="cell-summary"` to show each cell's source and measurements
immediately below it. The final cell displays the cumulative report.

In [ ]:
from linescope import profile

session = profile.start(memory=True, display="cell-summary")

In [ ]:
from collections import Counter
import re


def tokenize(document: str) -> list[str]:
    """Extract lowercase words from one document.

    Parameters
    ----------
    document : str
        Text to split into alphabetic words.

    Returns
    -------
    list[str]
        Lowercase words in their original order.

    """
    return re.findall(r"[a-z]+", document.lower())


documents = [
    f"Analysis {number}: notebooks keep Python source beside its results. "
    "A profiler helps find repeated parsing, sorting, and aggregation work."
    for number in range(300)
]
tokens = [tokenize(document) for document in documents]
frequencies = Counter(word for words in tokens for word in words)
common = frequencies.most_common(10)
unique_words = sorted(frequencies)
total = sum(frequencies.values())
print(f"Tokens: {total}; vocabulary: {len(unique_words)}")
print(common)

## One report for several cells

Continue the same session with the next two cells. Each compact summary shows
only that cell's measurements. The full overview at the end includes all cells.

In [ ]:
values = list(range(3000))
buckets = {}
for value in values:
    buckets.setdefault(value % 20, []).append(value * value)
squares = [square for group in buckets.values() for square in group]

In [ ]:
average = sum(squares) / len(squares)
summaries = {key: sum(group) / len(group) for key, group in buckets.items()}
largest = sorted(summaries.items(), key=lambda item: item[1], reverse=True)[:5]
print(f"Average square: {average:,.1f}; largest groups: {largest}")

## Finish and inspect the full overview

Stop collection and retrieve the normalized result through `profile.result`
(also available as `session.result`). `profile.stop()` returns None so it adds
no text output. Save the report and display it inline once, in the final cell.
Assign the HTML returned by `show()` so it does not print below the report.

Captured source stays in the report even if you later edit these cells. Review
source snapshots before sharing.

In [ ]:
profile.stop()
result = profile.result
report_path = session.save("notebook.html")
_ = session.show(inline=True)

## One report at the end

Use `display="end"` to keep ordinary cell outputs visible while collecting
measurements across cells. LineScope displays one complete report when you
call `profile.stop()`.

In [ ]:
from linescope import profile

end_session = profile.start(memory=True, display="end")

In [ ]:
values = list(range(3000))
squared = [value * value for value in values]
print(f"Prepared {len(squared):,} squares")

In [ ]:
mean_square = sum(squared) / len(squared)
print(f"Mean square: {mean_square:,.1f}")

In [ ]:
profile.stop()
end_result = profile.result
end_report_path = end_session.save("notebook-end.html")